# 03 — Constraints, Examples, and Few-Shot Learning

## Scenario
Northstar must route support messages to `refund`, `shipping`, `account`, or `unknown`. Ambiguous messages must remain `unknown`; a confident but incorrect category sends users into the wrong workflow.

**Safety boundary:** This notebook uses Northstar's replay/live client to retrieve safe, approved examples. Offline replay uses deterministic hash embeddings; live mode may use the configured provider's embedding service.

In [ ]:
from pathlib import Path
from northstar.contracts import parse_structured
from northstar.runtime import get_client, estimate_tokens, HASH_EMBEDDING_NOTICE
from lab03 import EVALUATION_SUITE, EXAMPLE_BANK, RoutingDecision, build_requests, select_examples, similarity_examples

client = get_client(Path("fixtures/replays.json"))
REQUESTS = {request.case_id: request for request in build_requests()}

def show(case_id):
    request = REQUESTS[case_id]
    print(f"\n=== {case_id} ===")
    print("SYSTEM:\n" + (request.system or "(none)"))
    for message in request.messages:
        print(f"{message.role.upper()}:\n{message.text}")
    response = client.generate(request)
    print("RECORDED RESPONSE:\n" + response.text)
    parsed = response.parsed or parse_structured(RoutingDecision, response.text).value
    print("PARSED VALUE:", parsed.model_dump())
    return request, response, parsed

## Baseline: Zero-Shot

We start with a direct instruction and no examples.

In [ ]:
expected = [case["expected"] for case in EVALUATION_SUITE]
observed = []
for case in EVALUATION_SUITE:
    _, _, parsed = show(f"b03/zero/{case['id']}")
    observed.append(parsed.category)
print("Zero-shot:", list(zip(expected, observed)))
assert sum(a == b for a, b in zip(expected, observed)) == 3

## Strategy 1: Static Few-Shot

We hardcode two examples into every prompt. This is predictable but uses context tokens on every request, even when the examples aren't relevant.

In [ ]:
observed = []
for case in EVALUATION_SUITE:
    _, _, parsed = show(f"b03/static/{case['id']}")
    observed.append(parsed.category)
print("Static examples:", list(zip([c["expected"] for c in EVALUATION_SUITE], observed)))
assert sum(a == b for a, b in zip([c["expected"] for c in EVALUATION_SUITE], observed)) == 4

## Strategy 2: Random Few-Shot Selection

We randomly select 2 examples from the bank. This provides diversity but risks injecting irrelevant or confusing examples.

In [ ]:
observed = []
for case in EVALUATION_SUITE:
    selected = select_examples(2, case["message"], EXAMPLE_BANK, f"b03/random/{case['id']}")
    print("Selected examples:", selected)
    assert all(example["message"] != case["message"] for example in selected)
    _, _, parsed = show(f"b03/random/{case['id']}")
    observed.append(parsed.category)
assert sum(a == b for a, b in zip([c["expected"] for c in EVALUATION_SUITE], observed)) == 3

## Strategy 3: Semantic Similarity Selection (RAG for Prompts)

The runtime embedding interface finds the 2 most relevant examples from the bank for the specific user query. Offline replay uses deterministic hash embeddings; live mode may use provider embeddings. Compare the selected context and token cost rather than assuming relevance.

In [ ]:
print(HASH_EMBEDDING_NOTICE)
observed = []
for case in EVALUATION_SUITE:
    selected = similarity_examples(client, case["message"], f"b03/similarity/{case['id']}")
    print("Embedding-selected examples:", selected)
    assert all(example["message"] != case["message"] for example in selected)
    _, _, parsed = show(f"b03/similarity/{case['id']}")
    observed.append(parsed.category)
assert sum(a == b for a, b in zip([c["expected"] for c in EVALUATION_SUITE], observed)) == 5
print("Estimated zero-shot tokens:", estimate_tokens(REQUESTS["b03/zero/clear-refund"].messages[0].text))

## Conclusion: Accuracy vs. Context Cost

By measuring both accuracy and context size (tokens), we can make an informed decision about our few-shot strategy.

In [ ]:
print("The recorded run compares accuracy with estimated context cost.")
for strategy in ("zero", "static", "random", "similarity"):
    tokens = sum(
        estimate_tokens(REQUESTS[f"b03/{strategy}/{case['id']}"].messages[0].text)
        for case in EVALUATION_SUITE
    )
    print(strategy, "estimated suite tokens:", tokens)

## Takeaway

The assertions above describe the deterministic recorded run. Change one prompt variable, rerun the lab, and measure the trade-off.

## References

See the course README for the folded reference material and links.

## Reading the example-selection experiment

The evaluation suite contains clear cases and deliberately ambiguous boundaries.
The expected labels are frozen before selection begins. Zero-shot is the baseline
for a direct instruction. Static few-shot selection is easy to audit because the
same refund and boundary examples appear for every query. Random selection is
useful as a robustness probe only when it is reproducible; this notebook seeds
selection with the case ID and explicitly removes a query if it appears in the
bank.

The leakage guard matters more than a superficially high score. If a query is
allowed to appear in its own examples, the model can copy the answer and the
evaluation no longer measures generalization. The bank also includes a negative
double-charge example, an account example, and shipping examples so that labels
represent both positive and boundary evidence.

The similarity strategy calls the runtime embedding interface. In replay mode the
vectors are deterministic lexical-hash vectors, so the notice printed above is
important: they are not semantic embeddings and should not be described as
meaning-aware retrieval. The point is to exercise the same selection interface
offline, compare measured outcomes, and make the limitation visible. A live
system would replace the vectors only after evaluating retrieval quality,
permission filters, freshness, and provenance.

The final table reports estimated context tokens alongside accuracy. More
examples can improve a boundary while increasing context cost and distracting
the model. The decision is therefore not “few-shot always wins”; it is to measure
the trade-off on a frozen suite and choose the smallest strategy that meets the
required quality. If the similarity ranking changes, inspect which examples
entered the prompt rather than only looking at the final accuracy number. Examples
are prompt data, not permissions: a real retriever still needs tenant, freshness,
and provenance filters before an example is shown to a model. A useful follow-up
is to remove one example, change only its label, and compare the resulting metric
and estimated prompt size. That is the same controlled-variable loop used by the
other beginner lessons.
Repeat it.
The exercise is intentionally small enough to repeat after every prompt edit.